In [ ]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate


# Map evaluation folder name suffixes (set by exp_name in the run scripts) to
# paper-style display names. Each suffix appears between the experiment prefix
# (e.g. "H16_1e6steps_") and the trailing "_<ode_t_steps>steps". Order matters:
# longer keys are checked first so "projection_all_gradient_guidance" wins
# over "projection_all".
DISPLAY_NAMES = [
    ("projection_all_gradient_guidance", "Projection-All + Gradient Guidance"),
    ("projection_late_gradient_guidance", "Projection-Late + Gradient Guidance"),
    ("projection_relaxed_gradient_guidance", "Projection-Relaxed + Gradient Guidance"),
    ("projection_all", "Projection-All"),
    ("projection_late", "Projection-Late"),
    ("projection_relaxed", "Projection-Relaxed"),
    ("hardflow_new", "HardFlow (l4casadi-free)"),
    ("hardflow", "HardFlow"),
    ("oc_flow", "OC-Flow"),
    ("gradient_guidance", "Gradient Guidance"),
    ("original", "Original"),
]


def get_display_name(experiment_name: str) -> str:
    for key, name in DISPLAY_NAMES:
        if key in experiment_name:
            return name
    return experiment_name


def analyze_experiment_results(env_name, filter_name=[None]):
    base_path = Path("../logs") / env_name / "eval"

    results = []

    for exp_dir in base_path.glob("*"):
        if not exp_dir.is_dir():
            continue

        csv_file = exp_dir / "trajectories.csv"
        if not csv_file.exists():
            continue

        not_match = False
        for n in filter_name:
            if n is not None and n not in exp_dir.name:
                not_match = True
                break
        if not_match:
            continue

        df_traj = pd.read_csv(csv_file)

        if df_traj.empty:
            continue

        total_trajs = len(df_traj)
        safety_trajs = df_traj["safety"].sum()
        safety_rate = safety_trajs / total_trajs if total_trajs > 0 else 0.0

        safe_df = df_traj[df_traj["safety"] == True]

        if len(safe_df) > 0:
            success_rate_safe = safe_df["success"].sum() / len(safe_df)
            steps_safe_mean = safe_df["steps"].mean()
            steps_safe_std = safe_df["steps"].std() if len(safe_df) > 1 else 0.0
        else:
            success_rate_safe = 0.0
            steps_safe_mean = 0.0
            steps_safe_std = 0.0

        if "average_computation_time" in df_traj.columns:
            average_computation_time = df_traj["average_computation_time"].dropna()
            if len(average_computation_time) > 0:
                computation_time_mean = average_computation_time.mean()
                computation_time_std = (
                    average_computation_time.std()
                    if len(average_computation_time) > 1
                    else 0.0
                )
                computation_time_available = True
            else:
                computation_time_mean = None
                computation_time_std = None
                computation_time_available = False
        else:
            computation_time_mean = None
            computation_time_std = None
            computation_time_available = False

        clean_name = exp_dir.name
        clean_name = clean_name.replace("H16_1e6steps_", "")
        display_name = get_display_name(clean_name)

        results.append(
            {
                "experiment": display_name,
                "safety_rate": safety_rate,
                "success_rate_safe": success_rate_safe,
                "steps_safe_mean": steps_safe_mean,
                "steps_safe_std": steps_safe_std,
                "computation_time_mean": computation_time_mean,
                "computation_time_std": computation_time_std,
                "computation_time_available": computation_time_available,
                "total_trajs": total_trajs,
                "safety_trajs": safety_trajs,
            }
        )

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values("safety_rate", ascending=False)

    return df


def display_results_table(df, header):
    if df.empty:
        print(f"\n{header} - No experiments found\n")
        return

    table_data = []
    for _, row in df.iterrows():
        if (
            row["computation_time_available"]
            and row["computation_time_mean"] is not None
        ):
            computation_time_str = (
                f"{row['computation_time_mean']:.3f}±{row['computation_time_std']:.3f}"
            )
        else:
            computation_time_str = "N/A"

        if row["safety_rate"] == 0:
            success_rate_str = "N/A"
            steps_str = "N/A"
        else:
            success_rate_str = f"{row['success_rate_safe']:.2f}"
            steps_str = f"{row['steps_safe_mean']:.2f}±{row['steps_safe_std']:.2f}"

        table_data.append(
            [
                row["experiment"],
                f"{row['safety_rate']:.2f}",
                success_rate_str,
                steps_str,
                computation_time_str,
                f"{row['total_trajs']}",
            ]
        )

    headers = [
        "Experiment",
        "Safety Rate",
        "Success Rate (Safety Trials)",
        "Steps (Safety Trials)",
        "Computation Time (s)",
        "Total Trials",
    ]

    print(f"\n{header} - {len(df)} experiments:")
    print(
        tabulate(
            table_data,
            headers=headers,
            tablefmt="grid",
            stralign="left",
            disable_numparse=True,
        )
    )


for env_name in ["avoiding-v0"]:
    results_df = analyze_experiment_results(env_name, filter_name=[""])
    if results_df.empty:
        continue

    print(f"\n{'=' * 120}")
    print(f"Results for {env_name}".center(120))
    print(f"{'=' * 120}")
    print(f"\nSummary: {len(results_df)} experiments")
    display_results_table(results_df, "Robotic Manipulation")
    print(f"\n{'=' * 120}")